# 01 · Chat templates + external tool + Pydantic answers

**What it tests:** single-prompt financial Q&A with the three Week 1 chat templates:
- zero-shot;
- structured few-shot;
- chain-of-thought.

The templates come from `scripts/evaluation/prompt_baselines.py`.

**Evidence: an external source, no RAG yet.**
1. The model is asked which **tool** to call, as JSON validated by the Pydantic `ToolCall` model.
2. The tool fetches the **exact figure from SEC's XBRL API** (the company's 10-K filing).
3. The figure is given to the LLM as evidence `[c1]`.

**Answer format:** every reply must parse into the Pydantic **`FinancialAnswer`** model (`scripts/agents/answer_schema.py`). Grounding is then checked: citations must exist in the evidence, and the value must appear in it.

**Questions:** taken from the frozen eval set (`data/evaluation/financial_report_qa/eval_set_v1.jsonl`): figures for 4 companies, plus not-disclosed and refusal items.

In [ ]:
# 0. Setup. On Colab: clones the repo (branch below) and installs packages. Locally: run from inside the repo.
import json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/rama159065/Capstone.git"
BRANCH = "finance_report_qa_agent"
IN_COLAB = "google.colab" in sys.modules


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if IN_COLAB:
    ROOT = Path("/content/Capstone")
    if not ROOT.exists():
        token = colab_secret("GITHUB_TOKEN")          # needed while the repo is private
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client>=1.12,<2", "pymupdf>=1.24,<2",
                    "pydantic>=2", "pandas", "sentence-transformers"], check=True)
    for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL", "SEC_USER_AGENT"):
        if not os.getenv(key) and colab_secret(key):
            os.environ[key] = colab_secret(key)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("SEC_USER_AGENT", "capstone-research your.name@example.com")   # SEC requires your real e-mail
print("repo:", ROOT, "| Colab:", IN_COLAB)

**LLM.** `NB_LLM=auto` picks, in order:
1. **The provided Track B endpoint**, when `LLM_API_BASE` and `LLM_MODEL` are set (Colab secrets or environment). Grade with this one.
2. **A small Hugging Face stand-in** (`Qwen/Qwen2.5-1.5B-Instruct`), on a Colab GPU.
3. **The deterministic mock** otherwise. It only proves that the pipeline runs.

Force a choice with `os.environ["NB_LLM"] = "hosted" | "transformers" | "mock"` before the config cell.

In [ ]:
from scripts.common import notebook_helpers as nb
from scripts.agents.answer_schema import FinancialAnswer, ToolCall, validate_answer
from scripts.agents.financial_clarifier import Clarifier
from scripts.evaluation import prompt_baselines as pb

COMPANIES = ["AMD_2025", "ABBV_2025", "NVDA_2026", "CVX_2022"]
PER_CATEGORY = {"single_figure": int(os.getenv("NB_FIGURES", "6")), "not_disclosed": 3, "refusal": 3}
llm, llm_name = nb.make_llm(os.getenv("NB_LLM", "auto"))
print("LLM:", llm_name)

## Pydantic schemas
`FinancialAnswer` is the answer contract; `ToolCall` is what the model may request before answering.

In [ ]:
import inspect
print(inspect.getsource(FinancialAnswer))
print(json.dumps(ToolCall.model_json_schema()["properties"], indent=1))

## External tool: exact XBRL figures from SEC

In [ ]:
for ticker, metric, year in [("AMD", "revenue", 2025), ("CVX", "diluted EPS", 2022), ("ABBV", "net income", 2025)]:
    fact = nb.get_xbrl_fact(ticker, metric, year)
    print(fact["text"] if fact else f"{ticker} {metric} {year}: not found")

## Step 1: the model chooses the tool (validated by Pydantic)

**If the model's JSON is not a valid `ToolCall`:** we record a `tool_call_error`, which is a Track B failure category. We then fall back to the deterministic slot parser that the chat API uses (`Clarifier.parse`).

**When the tool is `none`:** advice, predictions and instructions get no evidence. The template must then decline.

In [ ]:
TOOL_SYSTEM = ("You route financial questions to a tool. Return ONLY one JSON object, no prose: "
               '{"tool": "get_xbrl_fact" or "none", "ticker": "<upper-case ticker or null>", '
               '"metric": "<one of: ' + "; ".join(nb.METRICS) + '> or null", "fiscal_year": <integer or null>}. '
               'Use "none" for advice, predictions, instructions, or anything that is not a reported figure.')
clarifier = Clarifier()


def plan_tool_call(question):
    reply = llm.chat(TOOL_SYSTEM, f"Question: {question}\nReturn the JSON tool call.", temperature=0.0, max_tokens=80, seed=7)
    try:
        return ToolCall.model_validate(pb.parse_json(reply["text"]) or {}), "llm", None
    except Exception as exc:                       # pydantic ValidationError: unusable tool call
        error = "tool_call_error: " + str(exc).splitlines()[0]
    slots = clarifier.parse(question)              # deterministic fallback, same parser as the chat API
    if slots.companies and slots.years and slots.metric and slots.intent == "figure":
        return ToolCall(tool="get_xbrl_fact", ticker=slots.companies[0], metric=slots.metric,
                        fiscal_year=slots.years[0]), "fallback", error
    return ToolCall(tool="none"), "fallback", error


for q in ["What was AMD's revenue for fiscal 2025?", "Should I buy NVIDIA stock right now?"]:
    print(q, "->", plan_tool_call(q))

## Step 2: run every template on every question

In [ ]:
pool = nb.eval_items(COMPANIES)
items = [i for cat, n in PER_CATEGORY.items() for i in [x for x in pool if x["category"] == cat][:n]]
rows, replies = [], []
for item in items:
    call, source, tool_error = plan_tool_call(item["question"])
    fact = nb.get_xbrl_fact(call.ticker, call.metric, call.fiscal_year) if call.tool == "get_xbrl_fact" else None
    evidence = [fact] if fact else []
    for strategy in nb.TEMPLATES:
        reply = nb.run_template(llm, strategy, item["question"], evidence)
        model, problems = validate_answer(reply["parsed"], evidence)
        result = nb.score(item, reply, evidence)
        rows.append({"id": item["id"], "category": item["category"], "strategy": strategy,
                     "tool": f"{call.tool} ({source})", "tool_error": bool(tool_error), "evidence": len(evidence),
                     "correct": result["correct"], "failure": result["failure"],
                     "pydantic_valid": float(model is not None) if strategy != "zero_shot" else float("nan"),
                     "grounding_errors": ",".join(problems) if strategy != "zero_shot" else "",
                     "latency_s": round(reply["latency_s"], 2)})
        replies.append((item, strategy, reply))
results = nb.table(rows)
nb.show(results)

## Results

In [ ]:
nb.show(results.groupby("strategy").agg(accuracy=("correct", "mean"), pydantic_valid=("pydantic_valid", "mean"),
                                        tool_errors=("tool_error", "sum"), p50_latency_s=("latency_s", "median")).round(2))
nb.show(results.groupby(["category", "strategy"])["correct"].mean().unstack().round(2))
failed = results[~results.correct]
if len(failed):
    nb.show(failed.groupby(["strategy", "failure"]).size().unstack(fill_value=0))
nb.show(results[results.grounding_errors != ""].groupby(["strategy", "grounding_errors"]).size())

## One example of each failure

In [ ]:
seen = set()
for (item, strategy, reply), row in zip(replies, rows):
    key = (row["failure"], strategy)
    if row["failure"] and key not in seen:
        seen.add(key)
        print(f"--- {row['failure']} | {strategy} | {item['id']}: {item['question']}\nGOLD: {item['gold_answer'][:120]}\n"
              f"OUT:  {reply['text'][:400]!r}\nPydantic/grounding: {row['grounding_errors'] or ('n/a' if strategy == 'zero_shot' else 'ok')}\n")

## Failure scenarios this notebook exposes

| Failure | How it shows up here |
|---|---|
| **Tool-call error** | The model's routing JSON fails `ToolCall` (missing ticker or year, wrong metric name, prose around the JSON); counted in `tool_errors` |
| **Malformed output** | No JSON, or JSON that fails `FinancialAnswer`. Examples: an `answerable` reply without citations; a value given while declining. Chain-of-thought also gets cut off at the token limit |
| **Unsupported number** | `value_not_in_evidence`: the value isn't in the XBRL record. Common with a scale mix-up: the tool gives whole dollars, the model writes millions |
| **Wrong scale** | e.g. "$35,465" with no "million", or millions vs billions; scored `wrong_or_missing_scale` |
| **Copied few-shot example** | The answer talks about "Example Corp" or uses its numbers (seen in Week 1 for 40 of 86 questions) |
| **Permission / routing** | Advice or a forecast instead of declining; for prompt injection, doing what was asked instead of declining |
| **Over-refusal** | Declining although the tool returned the figure |

**Expected baseline:** Week 1 numbers for these templates are in `docs/week1_prompt_baselines.md`.